# enesys — Quickstart

**Forward-cost analysis of Germany's energy transition** — six paths, four lager-camps, 2026–2055.

This notebook walks through the model in five minutes:

1. **Forward-cost Rolling LCOE 2026-2055** for all six paths (neutral_default camp)
2. **Lager shift** — the LCOE winner changes across the four camps
3. **30-year trajectory** EE-GAS vs. KKW-GAS
4. **Tornado sensitivity** — which parameters move the result?
5. **Monte-Carlo** — distribution of EE-GAS vs. KKW-GAS over 500 runs

Repository: <https://github.com/berndhardung/enesys> · License: MIT

## Setup

**Colab / Binder:** install enesys directly from GitHub. **Local:** skip the install cell — `pip install -e ".[charts]"` from the repo root should already have given you `enesys`.

In [ ]:
# Auto-install on Colab / Binder, no-op locally.
try:
    import google.colab  # noqa: F401

    %pip install -q 'enesys[charts] @ git+https://github.com/berndhardung/enesys.git'
except ImportError:
    pass

In [ ]:
from enesys import (
    baseline_all_paths,
    compute_path,
    monte_carlo_all_paths,
    tornado_path_analysis,
)

CAMPS = ("neutral_default", "ee_optimistic", "atom_optimistic", "bestand_optimistic")
PATHS = ("weiterso", "bestand", "ee_gas", "ee_h2", "kkw_gas", "kkw_h2")

## 1. Forward-cost Rolling LCOE 2026-2055 — all six paths, neutral_default

Canonical trajectory aggregate: the 30-year Rolling mean over 2026-2055. EE-GAS, WEITER-SO and EE-H2 sit within 0.5 ct/kWh of each other; the forward-cost order is not stably determinable.

In [ ]:
prices = baseline_all_paths(camp="neutral_default")  # default = Rolling 2026-2055
print("Rolling LCOE 2026-2055, neutral_default (ct/kWh):\n")
for path, lcoe in sorted(prices.items(), key=lambda x: x[1]):
    print(f"  {path:<10} {lcoe:6.2f}")

## 2. Lager shift — the LCOE winner changes across the four camps

Under the four camp-defaults, the Rolling-LCOE winner across the full six-path set shifts substantially: `neutral_default` → EE-GAS, `ee_optimistic` → EE-GAS, `atom_optimistic` → KKW-GAS, `bestand_optimistic` → WEITER-SO. The active four-path competition (EE-GAS, EE-H2, KKW-GAS, KKW-H2) reorders such that no path holds the point-estimate slot across all four camps. A robust ranking therefore does **not** rest on point dominance — it rests on **min-max-regret** across the four camps (EE-Lager-Reue ~775 Mrd > KKW-Lager-Reue ~368 Mrd → EE-GAS minimax-regret-optimal).

In [ ]:
active = ("EE-GAS", "EE-H2", "KKW-GAS", "KKW-H2")
print(f"{'Camp':<22} " + " ".join(f"{p:>8}" for p in active) + "   Winner")
for camp in CAMPS:
    p = baseline_all_paths(camp=camp)  # Rolling 2026-2055
    winner = min(active, key=lambda k: p[k])
    row = " ".join(f"{p[k]:8.2f}" for k in active)
    print(f"{camp:<22} {row}   {winner}")

## 3. 30-year trajectory — EE-GAS vs. KKW-GAS

`compute_path` runs the full 2026 → 2055 trajectory. KKW-GAS carries the FID lead-time (3 a) plus realgrad-stretched build-time: in neutral_default the EPR comes online 2046, in atom_optimistic 2036, in ee_optimistic 2050. The bridge-phase before that runs on PV + wind + gas backup for both paths.

In [ ]:
years = list(range(2026, 2056))
ee = compute_path("ee_gas", years=years, camp="neutral_default")
kkw = compute_path("kkw_gas", years=years, camp="neutral_default")

print(f"{'Year':<6} {'EE-GAS':>8} {'KKW-GAS':>8}   ct/kWh")
for r_ee, r_kkw in zip(ee, kkw, strict=False):
    if r_ee.year in (2026, 2030, 2035, 2040, 2045, 2050, 2055):
        print(f"{r_ee.year:<6} {r_ee.lcoe_ct_kwh:8.2f} {r_kkw.lcoe_ct_kwh:8.2f}")

## 4. Tornado sensitivity — which parameters move the result?

Top hebel for both EE-GAS and KKW-GAS is the **NEP-Realisierungsgrad** (grid build-out). The min-operator couples policy realgrad and world-belief — in a NEP-skeptical world, EE policy is structurally throttled even in EE-paths.

In [ ]:
for path in ("ee_gas", "kkw_gas"):
    print(f"=== Tornado: {path} (top-5 hebel) ===")
    for r in tornado_path_analysis(path)[:5]:
        print(
            f"  {r['label']:<30} swing={r['swing']:5.2f} ct  "
            f"(low={r['price_low']:5.2f}, high={r['price_high']:5.2f})"
        )
    print()

## 5. Monte-Carlo — robustness over 500 runs

Monte-Carlo draws all parameters simultaneously from realistic distributions, calibrated so the median matches the neutral_default and the 5-/95-quantiles match the camp edges.

In [ ]:
mc = monte_carlo_all_paths(n_runs=500, seed=42)

print("Mean LCOE per path (ct/kWh, neutral_default Monte-Carlo):\n")
for path, mean in sorted(mc["mean_per_path"].items(), key=lambda x: x[1]):
    print(
        f"  {path:<10} mean={mean:6.2f}  std={mc['std_per_path'][path]:5.2f}  p5={mc['p5_per_path'][path]:6.2f}"
    )

print("\nP(EE-GAS wins vs ...):")
for other, p in sorted(mc["p_ee_gas_wins_vs"].items(), key=lambda x: -x[1]):
    print(f"  {other:<10}: {p * 100:5.1f} %")
print(f"\nEE-GAS in top-2: {mc['p_ee_gas_top2'] * 100:.1f} %")

## Further reading

- **Camp-symmetric sensitivity** — `docs/PARAM_SETS.md`
- **All formulas with derivation** — `docs/FORMULAS.md`
- **Every default with primary-source tag** — `docs/SOURCES.md`
- **Chart generators** — `examples/generate_chart_*.py` (run `make charts`)
- **Test suite** — `pytest tests/ -v`